In [1]:
import random, sqlite3, csv, os
random.seed(2604)
OUTDIR = os.getcwd()
CITIES = [
"Bengaluru"
,
"Mumbai"
,
"Delhi NCR"
,
"Pune"
,
"Hyderabad"
,
"Chennai"
]
ACTIVE_CATEGORIES = [
"AC Repair & Service"
,
"Salon for Women"
,
"Salon for Men"
,
"Deep Home Cleaning"
,
"Plumbing"
,
"Electrical Repair"
,]
HELD_OUT_CATEGORY ="Pest Control"
ALL_CATEGORIES = ACTIVE_CATEGORIES + [HELD_OUT_CATEGORY]
CATEGORY_PRICE_RANGE = {
"AC Repair & Service"
: (
499
,
2499
),
"Salon for Women"
: (
699
,
3499
),
"Salon for Men"
: (
349
,
1499
),
"Deep Home Cleaning"
: (
999
,
4999
),
"Plumbing"
: (
199
,
1499
),
"Electrical Repair"
: (
199
,
1999
),}
CATEGORY_WEIGHTS = [
0.22
,
0.20
,
0.14
,
0.18
,
0.14
,
0.12
]
PARTNERS = []
partner_seq =1
for city in CITIES:
    for _ in range (8):
        pid =f"P{partner_seq:03d}"
        cat = random.choices(ACTIVE_CATEGORIES, weights=CATEGORY_WEIGHTS, k=1)[0]
        rating =round(random.uniform(3.4,5.0),1)
        PARTNERS.append({
"partner_id"
: pid,
"city"
: city,
"primary_category"
: cat,
"rating"
: rating,
"active"
:
True
,
"days_since_onboarding"
: random.randint(
30
,
500
)})
        partner_seq +=1
IDLE_PARTNER_ID =f"P{partner_seq:03d}"# newly onboarded, zero bookings on purpose
PARTNERS.append({
"partner_id"
: IDLE_PARTNER_ID,
"city"
:
"Pune"
,
"primary_category"
:
"Plumbing"
,
"rating"
:
0.0
,
"active"
:
True
,
"days_since_onboarding"
:
4
})
partner_seq +=1
PARTNER_BY_ID = {p["partner_id"]: p for p in PARTNERS}
PARTNERS_BY_CITY = {}
for p in PARTNERS:
    if p["partner_id"] == IDLE_PARTNER_ID:
        continue
    PARTNERS_BY_CITY.setdefault(p["city"], []).append(p["partner_id"])
N_BOOKINGS =600
BOOKINGS = []
STATUS_CHOICES = ["Paid","Refunded","Pending"]
STATUS_WEIGHTS = [0.82,0.10,0.08]
for booking_seq in range(1, N_BOOKINGS +1):
    city = random.choice(CITIES)
    partner_id = random.choice(PARTNERS_BY_CITY[city])
    partner = PARTNER_BY_ID[partner_id]
    category = partner["primary_category"]
    lo, hi = CATEGORY_PRICE_RANGE[category]
    amount = random.randint(lo, hi)
    day = random.randint(1,90)
    month =1 if day <=30 else (2 if day <=60 else 3)
    day_in_month = day - (month -1) *30
    booking_date =f"2026-{month:02d}-{day_in_month:02d}"
    status = random.choices(STATUS_CHOICES, weights=STATUS_WEIGHTS, k=1)[0]
    complaint_flag =1 if random.random() < 0.12 else 0
    sla_breach_flag =1 if  random.random() < 0.15 else 0
    if status =="Pending":
        customer_rating =None
    else:
        base =4.3 - (1.4 if complaint_flag else 0) - (0.6 if sla_breach_flag else 0)
        customer_rating =max(1,min (5,round (base + random.uniform(-0.6,0.6))))
    is_test =1 if booking_seq in (37,214,501) else 0
    # status and customer_rating are computed above (to preserve the exact random-call
# sequence downstream) but deliberately not stored: neither is used by any task in
# this brief, so they are not written to bookings.csv/the bookings table.
    BOOKINGS.append({"booking_id":f"B{booking_seq:04d}","partner_id"
: partner_id,
"city"
: city,
"category"
: category,
"booking_date"
: booking_date,
"amount_inr"
: amount,
"complaint_flag"
: complaint_flag,
"sla_breach_flag"
: sla_breach_flag,
"is_test"
: is_test})
# ---- partners_import.csv: the ONLY partner file shipped -- a raw import with 3
# deliberate exact-duplicate rows. Deduplication is your own Part A task. ----
DUPLICATED_IDS = ["P003","P017","P031"]
import_rows =list(PARTNERS)
for pid in DUPLICATED_IDS:
    import_rows.append(dict(PARTNER_BY_ID[pid]))
random.shuffle(import_rows)
def write_csv(filename, rows, fieldnames):
    with open(os.path.join(OUTDIR, filename),"w", newline="")as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)
write_csv("cities.csv", [{"city": c} for c in CITIES], ["city"])
write_csv("categories.csv", [{"category": c}for c in ALL_CATEGORIES], ["category"])
write_csv("partners_import.csv", import_rows, [
"partner_id"
,
"city"
,
"primary_category"
,
"rating"
,
"active"
,
"days_since_onboarding"
])
write_csv("bookings.csv", BOOKINGS, [
"booking_id"
,
"partner_id"
,
"city"
,
"category"
,
"booking_date"
,
"amount_inr"
,
"complaint_flag"
,
"sla_breach_flag"
,
"is_test"
])

# ---- Load everything into a SQLite database ---

db_path = os.path.join(OUTDIR,"urban_service.db")
if os.path.exists(db_path):
    os.remove(db_path)
conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute("CREATE TABLE categories (category TEXT PRIMARY KEY)")
cur.executemany("INSERT INTO categories VALUES (?)", [(c,)for c in ALL_CATEGORIES])
cur.execute("""CREATE TABLE partners_import ( partner_id TEXT, city TEXT, primary_category TEXT, rating REAL, active INTEGER, days_since_onboarding INTEGER)""")
cur.executemany("INSERT INTO partners_import VALUES (?,?,?,?,?,?)", [(r["partner_id"], r["city"], r["primary_category"], r["rating"],1 if r["active"]else 0 , r["days_since_onboarding"]) for r in import_rows])
cur.execute("""CREATE TABLE bookings ( booking_id TEXT PRIMARY KEY, partner_id TEXT, city TEXT, category TEXT, booking_date TEXT, amount_inr INTEGER, complaint_flag INTEGER, sla_breach_flag INTEGER, is_test INTEGER)""")
cur.executemany(
"INSERT INTO bookings VALUES (?,?,?,?,?,?,?,?,?)"
, [(r[
"booking_id"
], r[
"partner_id"
], r[
"city"
], r[
"category"
], r[
"booking_date"
], r[
"amount_inr"
], r[
"complaint_flag"
], r[
"sla_breach_flag"
], r[
"is_test"
])
for
r
in
BOOKINGS])
conn.commit()
conn.close()
print("urban_service.db created with categories, partners_import, bookings tables.")

urban_service.db created with categories, partners_import, bookings tables.


step2:adding the files in the github


In [5]:
import sqlite3

connection = sqlite3.connect("urban_service.db")

print("categories:", connection.execute(
    "SELECT COUNT(*) FROM categories"
).fetchone()[0])

print("partners_import:", connection.execute(
    "SELECT COUNT(*) FROM partners_import"
).fetchone()[0])

print("bookings:", connection.execute(
    "SELECT COUNT(*) FROM bookings"
).fetchone()[0])

connection.close()

categories: 7
partners_import: 52
bookings: 600


In [6]:
verification = """-- SELECT COUNT(*) FROM categories;
-- Result: 7

-- SELECT COUNT(*) FROM partners_import;
-- Result: 52

-- SELECT COUNT(*) FROM bookings;
-- Result: 600
"""

with open("verify_output.txt", "w") as file:
    file.write(verification)

print("verify_output.txt created successfully.")

verify_output.txt created successfully.


In [7]:
print(open("verify_output.txt").read())

-- SELECT COUNT(*) FROM categories;
-- Result: 7

-- SELECT COUNT(*) FROM partners_import;
-- Result: 52

-- SELECT COUNT(*) FROM bookings;
-- Result: 600



In [8]:
%%writefile sanity_check.py
sample_bookings = [
    {
        "booking_id": "B0005",
        "category": "AC Repair & Service",
        "amount_inr": 1316
    },
    {
        "booking_id": "B0019",
        "category": "AC Repair & Service",
        "amount_inr": 538
    },
    {
        "booking_id": "B0027",
        "category": "AC Repair & Service",
        "amount_inr": 1016
    },
    {
        "booking_id": "B0055",
        "category": "AC Repair & Service",
        "amount_inr": 1505
    },
    {
        "booking_id": "B0001",
        "category": "Plumbing",
        "amount_inr": 1369
    },
    {
        "booking_id": "B0003",
        "category": "Plumbing",
        "amount_inr": 772
    },
    {
        "booking_id": "B0004",
        "category": "Plumbing",
        "amount_inr": 1133
    },
    {
        "booking_id": "B0006",
        "category": "Plumbing",
        "amount_inr": 805
    },
    {
        "booking_id": "B0018",
        "category": "Salon for Men",
        "amount_inr": 1414
    },
    {
        "booking_id": "B0024",
        "category": "Salon for Men",
        "amount_inr": 1176
    },
    {
        "booking_id": "B0029",
        "category": "Salon for Men",
        "amount_inr": 858
    },
    {
        "booking_id": "B0032",
        "category": "Salon for Men",
        "amount_inr": 638
    }
]

counts = {}
totals = {}

for booking in sample_bookings:
    category = booking["category"]
    amount = booking["amount_inr"]

    if category not in counts:
        counts[category] = 0
        totals[category] = 0

    counts[category] += 1
    totals[category] += amount

for category in counts:
    print(category, "count =", counts[category], "total =", totals[category])
# SQL result matches the pure-Python result exactly for all three categories.

Writing sanity_check.py


In [11]:
import sqlite3

conn = sqlite3.connect("urban_service.db")
cursor = conn.cursor()

query = """
SELECT category, COUNT(*), SUM(amount_inr)
FROM bookings
WHERE booking_id IN (
    'B0005',
    'B0019',
    'B0027',
    'B0055',
    'B0001',
    'B0003',
    'B0004',
    'B0006',
    'B0018',
    'B0024',
    'B0029',
    'B0032'
)
GROUP BY category;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

conn.close()

('AC Repair & Service', 4, 4375)
('Plumbing', 4, 4079)
('Salon for Men', 4, 4086)


In [12]:
with open("sanity_check.py", "a") as file:
    file.write("\n# SQL result matches the pure-Python result exactly for all three categories.\n")

print("Final comment added to sanity_check.py")

Final comment added to sanity_check.py


In [13]:
!tail -n 3 sanity_check.py

# SQL result matches the pure-Python result exactly for all three categories.

# SQL result matches the pure-Python result exactly for all three categories.


In [14]:
%%writefile 01_dedup_and_joins.sql

-- (a) List every duplicated partner_id
SELECT partner_id, COUNT(*) AS duplicate_count
FROM partners_import
GROUP BY partner_id
HAVING COUNT(*) > 1;


-- (b) Create a clean partners table by removing exact duplicate rows
CREATE TABLE partners AS
SELECT
    partner_id,
    city,
    primary_category,
    rating,
    active,
    days_since_onboarding
FROM partners_import
GROUP BY
    partner_id,
    city,
    primary_category,
    rating,
    active,
    days_since_onboarding;

Writing 01_dedup_and_joins.sql


In [15]:
!ls -l 01_dedup_and_joins.sql

-rw-r--r-- 1 root root 481 Sep 24 06:30 01_dedup_and_joins.sql


In [16]:
import sqlite3

conn = sqlite3.connect("urban_service.db")
cursor = conn.cursor()

# Check duplicate partner IDs
cursor.execute("""
SELECT partner_id, COUNT(*) AS duplicate_count
FROM partners_import
GROUP BY partner_id
HAVING COUNT(*) > 1;
""")

print("Duplicate partner IDs:")
for row in cursor.fetchall():
    print(row)

# Create clean partners table
cursor.execute("""
CREATE TABLE partners AS
SELECT
    partner_id,
    city,
    primary_category,
    rating,
    active,
    days_since_onboarding
FROM partners_import
GROUP BY
    partner_id,
    city,
    primary_category,
    rating,
    active,
    days_since_onboarding;
""")

conn.commit()

# Check clean table count
cursor.execute("SELECT COUNT(*) FROM partners;")
print("\nClean partners row count:", cursor.fetchone()[0])

conn.close()

Duplicate partner IDs:
('P003', 2)
('P017', 2)
('P031', 2)

Clean partners row count: 49


In [17]:
!ls -lh

total 136K
-rw-r--r-- 1 root root  481 Sep 24 06:30 01_dedup_and_joins.sql
-rw-r--r-- 1 root root  35K Sep 24 06:27 bookings.csv
-rw-r--r-- 1 root root  126 Sep 24 06:27 categories.csv
-rw-r--r-- 1 root root   62 Sep 24 06:27 cities.csv
-rw-r--r-- 1 root root 2.3K Sep 24 06:27 partners_import.csv
drwxr-xr-x 1 root root 4.0K Sep 16 13:26 sample_data
-rw-r--r-- 1 root root 1.9K Sep 24 06:30 sanity_check.py
-rw-r--r-- 1 root root  72K Sep 24 06:30 urban_service.db
-rw-r--r-- 1 root root  155 Sep 24 06:29 verify_output.txt


In [ ]:
!find /content -name "verify_output.txt" -o -name "sanity_check.py"

In [18]:
import os

print("Current folder:", os.getcwd())
print("\n/content contents:")
print(os.listdir("/content"))

Current folder: /content

/content contents:
['.config', 'sanity_check.py', 'bookings.csv', '01_dedup_and_joins.sql', 'verify_output.txt', 'cities.csv', 'categories.csv', 'partners_import.csv', 'urban_service.db', 'sample_data']


In [ ]:
%%writefile /content/verify_output.txt
-- SELECT COUNT(*) FROM categories;
-- Result: 7

-- SELECT COUNT(*) FROM partners_import;
-- Result: 52

-- SELECT COUNT(*) FROM bookings;
-- Result: 600

Writing /content/verify_output.txt


In [19]:
!cat /content/verify_output.txt


-- SELECT COUNT(*) FROM categories;
-- Result: 7

-- SELECT COUNT(*) FROM partners_import;
-- Result: 52

-- SELECT COUNT(*) FROM bookings;
-- Result: 600


In [20]:
%%writefile /content/sanity_check.py

sample_bookings = [
    {"booking_id": "B0005", "category": "AC Repair & Service", "amount_inr": 1316},
    {"booking_id": "B0019", "category": "AC Repair & Service", "amount_inr": 538},
    {"booking_id": "B0027", "category": "AC Repair & Service", "amount_inr": 1016},
    {"booking_id": "B0055", "category": "AC Repair & Service", "amount_inr": 1505},

    {"booking_id": "B0001", "category": "Plumbing", "amount_inr": 1369},
    {"booking_id": "B0003", "category": "Plumbing", "amount_inr": 772},
    {"booking_id": "B0004", "category": "Plumbing", "amount_inr": 1133},
    {"booking_id": "B0006", "category": "Plumbing", "amount_inr": 805},

    {"booking_id": "B0018", "category": "Salon for Men", "amount_inr": 1414},
    {"booking_id": "B0024", "category": "Salon for Men", "amount_inr": 1176},
    {"booking_id": "B0029", "category": "Salon for Men", "amount_inr": 858},
    {"booking_id": "B0032", "category": "Salon for Men", "amount_inr": 638}
]

counts = {}
totals = {}

for booking in sample_bookings:
    category = booking["category"]
    amount = booking["amount_inr"]

    if category not in counts:
        counts[category] = 0
        totals[category] = 0

    counts[category] += 1
    totals[category] += amount

for category in counts:
    print(category, "count =", counts[category], "total =", totals[category])

# SQL result matches the pure-Python result exactly for all three categories.

Overwriting /content/sanity_check.py


In [21]:
!python /content/sanity_check.py

AC Repair & Service count = 4 total = 4375
Plumbing count = 4 total = 4079
Salon for Men count = 4 total = 4086


In [74]:
import sqlite3

sql_file_1 = """-- Task 4(a): Identify duplicate partner records
SELECT partner_id, COUNT(*) AS dup_count
FROM partners_import
GROUP BY partner_id
HAVING COUNT(*) > 1;

-- Task 4(b): Deduplicate into clean partners table
CREATE TABLE partners AS
SELECT partner_id, city, primary_category, rating, active, days_since_onboarding
FROM partners_import
GROUP BY partner_id, city, primary_category, rating, active, days_since_onboarding;

-- Task 5(a): Confirm all bookings resolve to a real partner
SELECT COUNT(*) AS matched_bookings
FROM bookings b
INNER JOIN partners p ON b.partner_id = p.partner_id;

-- Task 5(b): Categories with zero bookings
SELECT c.category
FROM categories c
LEFT JOIN bookings b ON c.category = b.category
WHERE b.booking_id IS NULL;

-- Task 5(c): Partners with zero bookings
SELECT p.partner_id, p.city, p.primary_category
FROM partners p
LEFT JOIN bookings b ON p.partner_id = b.partner_id
WHERE b.booking_id IS NULL;

-- Task 5(d): Category-level COUNT(*) vs COUNT(b.booking_id)
-- COUNT(*) counts the joined row itself including the all-NULL unmatched row
-- while COUNT(b.booking_id) only counts rows where a real booking matched.
SELECT
    c.category,
    COUNT(*) AS total_rows,
    COUNT(b.booking_id) AS matched_bookings
FROM categories c
LEFT JOIN bookings b ON c.category = b.category
GROUP BY c.category;
"""

# Recreate the SQL file
with open("01_dedup_and_joins.sql", "w", encoding="utf-8") as f:
    f.write(sql_file_1)

# Recreate clean partners table
conn = sqlite3.connect("urban_service.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS partners")

# Execute the SQL file
cur.executescript(sql_file_1)

# Confirm clean partners count
cur.execute("SELECT COUNT(*) FROM partners")
print("Clean partners count:", cur.fetchone()[0])

conn.close()

Clean partners count: 49


In [76]:
print(open("01_dedup_and_joins.sql", encoding="utf-8").read())

-- Task 4(a): Identify duplicate partner records
SELECT partner_id, COUNT(*) AS dup_count
FROM partners_import
GROUP BY partner_id
HAVING COUNT(*) > 1;

-- Task 4(b): Deduplicate into clean partners table
CREATE TABLE partners AS
SELECT partner_id, city, primary_category, rating, active, days_since_onboarding
FROM partners_import
GROUP BY partner_id, city, primary_category, rating, active, days_since_onboarding;

-- Task 5(a): Confirm all bookings resolve to a real partner
SELECT COUNT(*) AS matched_bookings
FROM bookings b
INNER JOIN partners p ON b.partner_id = p.partner_id;

-- Task 5(b): Categories with zero bookings
SELECT c.category
FROM categories c
LEFT JOIN bookings b ON c.category = b.category
WHERE b.booking_id IS NULL;

-- Task 5(c): Partners with zero bookings
SELECT p.partner_id, p.city, p.primary_category
FROM partners p
LEFT JOIN bookings b ON p.partner_id = b.partner_id
WHERE b.booking_id IS NULL;

-- Task 5(d): Category-level COUNT(*) vs COUNT(b.booking_id)
-- COUNT(*

In [77]:
import sqlite3

conn = sqlite3.connect("urban_service.db")
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM partners")
count = cur.fetchone()[0]

print("Partners table rows:", count)

conn.close()

Partners table rows: 49


In [66]:
import sqlite3

conn = sqlite3.connect("/content/urban_service.db")
cursor = conn.cursor()

# (a) Every booking should resolve to a real partner
cursor.execute("""
SELECT b.booking_id, b.partner_id
FROM bookings b
INNER JOIN partners p
    ON b.partner_id = p.partner_id;
""")
result_a = cursor.fetchall()

print("(a) INNER JOIN")
print("Bookings resolved to partners:", len(result_a))


# (b) Categories with no bookings
cursor.execute("""
SELECT c.category
FROM categories c
LEFT JOIN bookings b
    ON c.category = b.category
WHERE b.booking_id IS NULL;
""")
result_b = cursor.fetchall()

print("\n(b) Categories with no bookings:")
print(result_b)


# (c) Partners with no bookings
cursor.execute("""
SELECT p.partner_id
FROM partners p
LEFT JOIN bookings b
    ON p.partner_id = b.partner_id
WHERE b.booking_id IS NULL;
""")
result_c = cursor.fetchall()

print("\n(c) Partners with no bookings:")
print(result_c)


# (d) COUNT(*) vs COUNT(b.booking_id)
cursor.execute("""
SELECT
    c.category,
    COUNT(*) AS total_joined_rows,
    COUNT(b.booking_id) AS booking_count
FROM categories c
LEFT JOIN bookings b
    ON c.category = b.category
GROUP BY c.category;
""")
result_d = cursor.fetchall()

print("\n(d) Category counts:")
for row in result_d:
    print(row)

conn.close()

(a) INNER JOIN
Bookings resolved to partners: 600

(b) Categories with no bookings:
[('Pest Control',)]

(c) Partners with no bookings:
[('P049',)]

(d) Category counts:
('AC Repair & Service', 74, 74)
('Deep Home Cleaning', 176, 176)
('Electrical Repair', 113, 113)
('Pest Control', 1, 0)
('Plumbing', 94, 94)
('Salon for Men', 78, 78)
('Salon for Women', 65, 65)


In [50]:
%%writefile /content/02_insert_delete.sql

-- (a) Delete the 3 dummy/test bookings
DELETE FROM bookings
WHERE is_test = 1;


-- (b) Insert exactly these 3 new bookings
INSERT INTO bookings
VALUES
(
    'B9001',
    'P009',
    'Mumbai',
    'Deep Home Cleaning',
    '2026-03-31',
    3200,
    'Paid',
    5,
    0,
    0,
    0
),
(
    'B9002',
    'P041',
    'Chennai',
    'Plumbing',
    '2026-03-31',
    640,
    'Paid',
    4,
    0,
    0,
    0
),
(
    'B9003',
    'P035',
    'Hyderabad',
    'Electrical Repair',
    '2026-03-31',
    980,
    'Pending',
    NULL,
    0,
    0,
    0
);


-- Verification:
-- SELECT COUNT(*), SUM(amount_inr) FROM bookings;
-- Expected: 600 rows, ₹10,47,973 total

Writing /content/02_insert_delete.sql


In [51]:
!cat /content/02_insert_delete.sql


-- (a) Delete the 3 dummy/test bookings
DELETE FROM bookings
WHERE is_test = 1;


-- (b) Insert exactly these 3 new bookings
INSERT INTO bookings
VALUES
(
    'B9001',
    'P009',
    'Mumbai',
    'Deep Home Cleaning',
    '2026-03-31',
    3200,
    'Paid',
    5,
    0,
    0,
    0
),
(
    'B9002',
    'P041',
    'Chennai',
    'Plumbing',
    '2026-03-31',
    640,
    'Paid',
    4,
    0,
    0,
    0
),
(
    'B9003',
    'P035',
    'Hyderabad',
    'Electrical Repair',
    '2026-03-31',
    980,
    'Pending',
    NULL,
    0,
    0,
    0
);


-- Verification:
-- SELECT COUNT(*), SUM(amount_inr) FROM bookings;
-- Expected: 600 rows, ₹10,47,973 total


In [53]:
import sqlite3

conn = sqlite3.connect("/content/urban_service.db")
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(bookings);")

for row in cursor.fetchall():
    print(row)

conn.close()

(0, 'booking_id', 'TEXT', 0, None, 1)
(1, 'partner_id', 'TEXT', 0, None, 0)
(2, 'city', 'TEXT', 0, None, 0)
(3, 'category', 'TEXT', 0, None, 0)
(4, 'booking_date', 'TEXT', 0, None, 0)
(5, 'amount_inr', 'INTEGER', 0, None, 0)
(6, 'complaint_flag', 'INTEGER', 0, None, 0)
(7, 'sla_breach_flag', 'INTEGER', 0, None, 0)
(8, 'is_test', 'INTEGER', 0, None, 0)


In [59]:
import csv
import sqlite3

sql_file_2 = """-- Task 6(a): Delete dummy/test rows
DELETE FROM bookings WHERE is_test = 1;

-- Task 6(b): Insert 3 operational bookings
INSERT INTO bookings VALUES
('B9001', 'P009', 'Mumbai', 'Deep Home Cleaning', '2026-03-31', 3200, 0, 0, 0),
('B9002', 'P041', 'Chennai', 'Plumbing', '2026-03-31', 640, 0, 0, 0),
('B9003', 'P035', 'Hyderabad', 'Electrical Repair', '2026-03-31', 980, 0, 0, 0);

-- Task 7: LIKE query for Salon partners
SELECT partner_id, primary_category, city, rating
FROM partners
WHERE primary_category LIKE 'Salon%';
"""

with open("02_insert_delete.sql", "w") as f:
  f.write(sql_file_2)

conn = sqlite3.connect("urban_service.db")
cur = conn.cursor()
cur.executescript(sql_file_2)

# Check Task 6 post-modification count and sum
cur.execute("SELECT COUNT(*), SUM(amount_inr) FROM bookings")
count, total_sum = cur.fetchone()
print(f"Post-mod Bookings: Count = {count}, Total = ₹{total_sum}")
# Target acceptance: exactly 600 rows, ₹10,47,973 total

# Task 8: Export city_category_summary.csv
summary_query = """
SELECT
    city,
    category,
    COUNT(*) AS bookings_count,
    SUM(amount_inr) AS revenue_inr,
    SUM(sla_breach_flag) AS sla_breaches
FROM bookings
GROUP BY city, category
ORDER BY city, category;
"""
cur.execute(summary_query)
rows = cur.fetchall()
col_names = [d[0] for d in cur.description]

with open("city_category_summary.csv", "w", newline="") as f:
  writer = csv.writer(f)
  writer.writerow(col_names)
  writer.writerows(rows)

print(f"Exported city_category_summary.csv with {len(rows)} data rows.")
conn.close()

IntegrityError: UNIQUE constraint failed: bookings.booking_id

In [60]:
import csv
import sqlite3

sql_file_2 = """-- Task 6(a): Delete dummy/test rows
DELETE FROM bookings WHERE is_test = 1;

-- Task 6(b): Insert 3 operational bookings
INSERT INTO bookings VALUES
('B9001', 'P009', 'Mumbai', 'Deep Home Cleaning', '2026-03-31', 3200, 0, 0, 0),
('B9002', 'P041', 'Chennai', 'Plumbing', '2026-03-31', 640, 0, 0, 0),
('B9003', 'P035', 'Hyderabad', 'Electrical Repair', '2026-03-31', 980, 0, 0, 0);

-- Task 7: LIKE query for Salon partners
SELECT partner_id, primary_category, city, rating
FROM partners
WHERE primary_category LIKE 'Salon%';
"""

with open("02_insert_delete.sql", "w") as f:
  f.write(sql_file_2)

conn = sqlite3.connect("urban_service.db")
cur = conn.cursor()
cur.executescript(sql_file_2)

# Check Task 6 post-modification count and sum
cur.execute("SELECT COUNT(*), SUM(amount_inr) FROM bookings")
count, total_sum = cur.fetchone()
print(f"Post-mod Bookings: Count = {count}, Total = ₹{total_sum}")
# Target acceptance: exactly 600 rows, ₹10,47,973 total

# Task 8: Export city_category_summary.csv
summary_query = """
SELECT
    city,
    category,
    COUNT(*) AS bookings_count,
    SUM(amount_inr) AS revenue_inr,
    SUM(sla_breach_flag) AS sla_breaches
FROM bookings
GROUP BY city, category
ORDER BY city, category;
"""
cur.execute(summary_query)
rows = cur.fetchall()
col_names = [d[0] for d in cur.description]

with open("city_category_summary.csv", "w", newline="") as f:
  writer = csv.writer(f)
  writer.writerow(col_names)
  writer.writerows(rows)

print(f"Exported city_category_summary.csv with {len(rows)} data rows.")
conn.close()

IntegrityError: UNIQUE constraint failed: bookings.booking_id

In [61]:
print(sql_file_2)

-- Task 6(a): Delete dummy/test rows
DELETE FROM bookings WHERE is_test = 1;

-- Task 6(b): Insert 3 operational bookings
INSERT INTO bookings VALUES
('B9001', 'P009', 'Mumbai', 'Deep Home Cleaning', '2026-03-31', 3200, 0, 0, 0),
('B9002', 'P041', 'Chennai', 'Plumbing', '2026-03-31', 640, 0, 0, 0),
('B9003', 'P035', 'Hyderabad', 'Electrical Repair', '2026-03-31', 980, 0, 0, 0);

-- Task 7: LIKE query for Salon partners
SELECT partner_id, primary_category, city, rating
FROM partners
WHERE primary_category LIKE 'Salon%';



In [63]:
import sqlite3

conn = sqlite3.connect("urban_service.db")
cur = conn.cursor()

cur.execute("SELECT COUNT(*), SUM(amount_inr) FROM bookings")
print("Bookings:", cur.fetchone())

cur.execute("""
SELECT *
FROM bookings
WHERE booking_id IN ('B9001', 'B9002', 'B9003')
ORDER BY booking_id
""")

print("Inserted bookings:")
for row in cur.fetchall():
    print(row)

conn.close()

Bookings: (600, 1047973)
Inserted bookings:
('B9001', 'P009', 'Mumbai', 'Deep Home Cleaning', '2026-03-31', 3200, 0, 0, 0)
('B9002', 'P041', 'Chennai', 'Plumbing', '2026-03-31', 640, 0, 0, 0)
('B9003', 'P035', 'Hyderabad', 'Electrical Repair', '2026-03-31', 980, 0, 0, 0)


In [64]:
sql_file_2 = """-- Task 6(a): Delete dummy/test rows
DELETE FROM bookings WHERE is_test = 1;

-- Task 6(b): Insert 3 operational bookings
INSERT INTO bookings VALUES
('B9001', 'P009', 'Mumbai', 'Deep Home Cleaning', '2026-03-31', 3200, 0, 0, 0),
('B9002', 'P041', 'Chennai', 'Plumbing', '2026-03-31', 640, 0, 0, 0),
('B9003', 'P035', 'Hyderabad', 'Electrical Repair', '2026-03-31', 980, 0, 0, 0);

-- 600 rows, ₹10,47,973 total

-- Task 7: LIKE query for Salon partners
SELECT partner_id, primary_category, city, rating
FROM partners
WHERE primary_category LIKE 'Salon%';
"""

with open("02_insert_delete.sql", "w", encoding="utf-8") as f:
    f.write(sql_file_2)

print("✅ File updated successfully")

✅ File updated successfully


In [65]:
print(sql_file_2)

-- Task 6(a): Delete dummy/test rows
DELETE FROM bookings WHERE is_test = 1;

-- Task 6(b): Insert 3 operational bookings
INSERT INTO bookings VALUES
('B9001', 'P009', 'Mumbai', 'Deep Home Cleaning', '2026-03-31', 3200, 0, 0, 0),
('B9002', 'P041', 'Chennai', 'Plumbing', '2026-03-31', 640, 0, 0, 0),
('B9003', 'P035', 'Hyderabad', 'Electrical Repair', '2026-03-31', 980, 0, 0, 0);

-- 600 rows, ₹10,47,973 total

-- Task 7: LIKE query for Salon partners
SELECT partner_id, primary_category, city, rating
FROM partners
WHERE primary_category LIKE 'Salon%';



In [85]:
import sqlite3

conn = sqlite3.connect("urban_service.db")
cur = conn.cursor()

cur.execute("""
SELECT partner_id, primary_category, city, rating
FROM partners
WHERE primary_category LIKE 'Salon%';
""")

results = cur.fetchall()

for row in results:
    print(row)

conn.close()


('P002', 'Salon for Men', 'Bengaluru', 4.9)
('P008', 'Salon for Women', 'Bengaluru', 3.8)
('P010', 'Salon for Men', 'Mumbai', 4.9)
('P015', 'Salon for Women', 'Mumbai', 4.7)
('P016', 'Salon for Men', 'Mumbai', 4.3)
('P023', 'Salon for Women', 'Delhi NCR', 3.7)
('P026', 'Salon for Men', 'Pune', 3.6)
('P027', 'Salon for Women', 'Pune', 4.3)
('P031', 'Salon for Women', 'Pune', 3.6)
('P034', 'Salon for Women', 'Hyderabad', 4.8)
('P036', 'Salon for Men', 'Hyderabad', 4.6)
('P040', 'Salon for Men', 'Hyderabad', 4.6)


In [86]:
import csv
import sqlite3

conn = sqlite3.connect("urban_service.db")
cur = conn.cursor()

summary_query = """
SELECT
    city,
    category,
    COUNT(*) AS bookings_count,
    SUM(amount_inr) AS revenue_inr,
    SUM(sla_breach_flag) AS sla_breaches
FROM bookings
GROUP BY city, category
ORDER BY city, category;
"""

cur.execute(summary_query)
rows = cur.fetchall()
col_names = [d[0] for d in cur.description]

with open("city_category_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(col_names)
    writer.writerows(rows)

print(f"✅ Exported city_category_summary.csv with {len(rows)} data rows.")

conn.close()

✅ Exported city_category_summary.csv with 27 data rows.


In [87]:
import csv
import openpyxl
from openpyxl.styles import Font, PatternFill

wb = openpyxl.Workbook()

# ----------------------------------------------------
# 1. Sheet: City-Category Data
# ----------------------------------------------------
ws_data = wb.active
ws_data.title = "City-Category Data"

with open("city_category_summary.csv", mode="r") as f:
  reader = csv.reader(f)
  for row in reader:
    # Convert numeric fields to int
    converted_row = []
    for val in row:
      try:
        converted_row.append(int(val))
      except ValueError:
        converted_row.append(val)
    ws_data.append(converted_row)

# Add VLOOKUP Headers
ws_data["F1"] = "min_price_inr"
ws_data["G1"] = "max_price_inr"

# 27 data rows (rows 2 to 28)
for r in range(2, 29):
  ws_data[f"F{r}"] = (
      f"=VLOOKUP(B{r}, 'Category Reference'!$A$2:$C$7, 2, FALSE)"
  )
  ws_data[f"G{r}"] = (
      f"=VLOOKUP(B{r}, 'Category Reference'!$A$2:$C$7, 3, FALSE)"
  )

# ----------------------------------------------------
# 2. Sheet: Category Reference
# ----------------------------------------------------
ws_ref = wb.create_sheet(title="Category Reference")
cat_data = [
    ["category", "min_price_inr", "max_price_inr"],
    ["AC Repair & Service", 499, 2499],
    ["Salon for Women", 699, 3499],
    ["Salon for Men", 349, 1499],
    ["Deep Home Cleaning", 999, 4999],
    ["Plumbing", 199, 1499],
    ["Electrical Repair", 199, 1999],
]
for row in cat_data:
  ws_ref.append(row)

# ----------------------------------------------------
# 3. Sheet: KPI Summary
# ----------------------------------------------------
ws_kpi = wb.create_sheet(title="KPI Summary")
kpi_headers = [
    "city",
    "total_revenue",
    "total_sla_breaches",
    "city_category_count",
    "Part A SQL Total",
    "Matches Part A SQL total?",
]
ws_kpi.append(kpi_headers)

cities_targets = [
    ("Bengaluru", 179835),
    ("Chennai", 175572),
    ("Delhi NCR", 140771),
    ("Hyderabad", 171638),
    ("Mumbai", 151430),
    ("Pune", 228727),
]

green_fill = PatternFill(
    start_color="C6EFCE", end_color="C6EFCE", fill_type="solid"
)
red_fill = PatternFill(
    start_color="FFC7CE", end_color="FFC7CE", fill_type="solid"
)
header_font = Font(bold=True)

for cell in ws_kpi[1]:
  cell.font = header_font

for idx, (city, target) in enumerate(cities_targets, start=2):
  ws_kpi[f"A{idx}"] = city
  ws_kpi[f"B{idx}"] = (
      f"=SUMIFS('City-Category Data'!$D$2:$D$28, 'City-Category Data'!$A$2:$A$28,"
      f' "{city}")'
  )
  ws_kpi[f"C{idx}"] = (
      f"=SUMIFS('City-Category Data'!$E$2:$E$28, 'City-Category Data'!$A$2:$A$28,"
      f' "{city}")'
  )
  ws_kpi[f"D{idx}"] = (
      f'=COUNTIFS(\'City-Category Data\'!$A$2:$A$28, "{city}")'
  )
  ws_kpi[f"E{idx}"] = target
  ws_kpi[f"F{idx}"] = f'=IF(B{idx}=E{idx}, "Yes", "No")'

  # Highlight highest (Pune) and lowest (Delhi NCR) revenue cities
  if city == "Pune":
    ws_kpi[f"B{idx}"].fill = green_fill
  elif city == "Delhi NCR":
    ws_kpi[f"B{idx}"].fill = red_fill

wb.save("urban_company_metrics.xlsx")
print(
    "urban_company_metrics.xlsx created with City-Category Data, Category"
    " Reference, and KPI Summary."
)

urban_company_metrics.xlsx created with City-Category Data, Category Reference, and KPI Summary.


In [89]:
import csv

with open("city_category_summary.csv", encoding="utf-8") as f:
    rows = list(csv.reader(f))

print("Total rows including header:", len(rows))
print("Data rows:", len(rows) - 1)
print("Header:", rows[0])

Total rows including header: 28
Data rows: 27
Header: ['city', 'category', 'bookings_count', 'revenue_inr', 'sla_breaches']


In [98]:
import os
import shutil

drive_folder = "/content/drive/MyDrive/Urban_Service_Assignment"
os.makedirs(drive_folder, exist_ok=True)

files = [
    "generate_data.py",
    "urban_service.db",
    "cities.csv",
    "categories.csv",
    "partners_import.csv",
    "bookings.csv",
    "verify_output.txt",
    "01_dedup_and_joins.sql",
    "02_insert_delete.sql",
    "city_category_summary.csv",
    "urban_company_metrics.xlsx"
]

for file in files:
    if os.path.exists(file):
        shutil.copy2(file, drive_folder)
        print("Copied:", file)
    else:
        print("Not found:", file)

print("\nDone! Folder:", drive_folder)

Copied: generate_data.py
Copied: urban_service.db
Copied: cities.csv
Copied: categories.csv
Copied: partners_import.csv
Copied: bookings.csv
Copied: verify_output.txt
Copied: 01_dedup_and_joins.sql
Copied: 02_insert_delete.sql
Copied: city_category_summary.csv
Copied: urban_company_metrics.xlsx

Done! Folder: /content/drive/MyDrive/Urban_Service_Assignment
